# MVP - Machine Learning & Analytics: Previsao de Chuva na Australia

**Nome:** Henrique Teixeira
**Dataset:** Rain in Australia (weatherAUS) - Australian Bureau of Meteorology
**Tipo de problema:** Classificacao binaria supervisionada

---

Este notebook constroi e avalia modelos de classificacao para prever a ocorrencia de chuva no dia seguinte (`RainTomorrow`) a partir de observacoes meteorologicas diarias registradas em estacoes australianas. O trabalho segue um fluxo completo de Machine Learning: definicao do problema, analise exploratoria, preparacao dos dados com prevencao de vazamento, modelagem, otimizacao de hiperparametros e discussao critica dos resultados.

## Checklist do MVP

| Item | Status |
|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | OK |
| Dataset descrito, com fonte, atributos e restricoes | OK |
| Dataset carregado por URL publica | OK |
| Analise exploratoria conectada a modelagem | pendente |
| Divisao adequada em treino/teste | pendente |
| Prevencao de vazamento de dados | pendente |
| Tratamentos justificados | pendente |
| Pipeline reprodutivel de pre-processamento | pendente |
| Modelo baseline definido | pendente |
| Pelo menos dois modelos candidatos alem do template | pendente |
| Ajuste de hiperparametros em pelo menos um modelo | pendente |
| Avaliacao com metricas coerentes | pendente |
| Discussao de overfitting/underfitting e limitacoes | pendente |
| Codigo limpo e executavel do inicio ao fim | pendente |
| Conclusao conectada ao objetivo | pendente |

# 1. Definicao do problema

## 1.1 Contexto e objetivo

Prever se chovera no dia seguinte tem valor operacional direto em agricultura, logistica, gestao de recursos hidricos e planejamento de atividades ao ar livre. O dataset weatherAUS reune cerca de dez anos de observacoes meteorologicas diarias de 49 estacoes espalhadas pela Australia, com variaveis como temperatura, umidade, pressao, vento e nebulosidade medidas em dois horarios do dia (9h e 15h).

O objetivo deste MVP e construir e avaliar modelos que, a partir das condicoes meteorologicas observadas em um dia, prevejam a variavel binaria `RainTomorrow` (choveu no dia seguinte: sim ou nao), comparando um baseline ingenuo com modelos candidatos e discutindo criticamente seus limites.

## 1.2 Por que e um problema de Machine Learning

A relacao entre as condicoes atmosfericas de um dia e a ocorrencia de chuva no dia seguinte e complexa, nao linear e envolve interacao entre multiplas variaveis. Nao existe regra deterministica simples que resolva o problema, mas existe um padrao estatistico aprendivel a partir de dados historicos. Isso caracteriza um problema supervisionado de classificacao binaria.

## 1.3 Tipo de problema e metrica

**Tipo:** classificacao binaria supervisionada.

O target apresenta desbalanceamento real: aproximadamente 22% dos dias sao seguidos de chuva contra 78% sem chuva. Esse desbalanceamento tem consequencia direta na escolha de metricas. Acuracia seria enganosa, um modelo que sempre preve "nao chove" atingiria cerca de 78% de acuracia sem aprender nada util. Por isso a metrica principal sera o F1-score da classe positiva (chuva) e a PR-AUC (area sob a curva precision-recall), que avaliam a capacidade real de identificar os dias de chuva. A analise sera complementada com estudo de threshold.

## 1.4 Premissas e restricoes

As observacoes sao tratadas como diarias e independentes no nivel de modelagem tabular, embora exista estrutura temporal que sera respeitada na divisao treino/teste. Assume-se que os padroes meteorologicos aprendidos no periodo de treino permanecem validos no periodo de teste. A principal restricao de dados, discutida adiante, e a presenca da coluna `RISK_MM`, que constitui vazamento direto do target e sera removida.

# 2. Ambiente e reprodutibilidade

XGBoost e LightGBM sao instalados de forma idempotente. No Google Colab o XGBoost costuma vir pre-instalado, mas a instalacao explicita blinda o notebook contra variacoes da imagem do ambiente, garantindo execucao do inicio ao fim.

In [ ]:
!pip install -q xgboost lightgbm

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", None)

print("Python:", sys.version.split()[0])
print("Seed:", SEED)

# 3. Carga dos dados

O dataset e carregado diretamente da versao raw do repositorio publico no GitHub, sem upload manual, login ou token. Isso garante que o notebook execute do inicio ao fim em qualquer ambiente com acesso a internet.

A coluna `Date` e lida no formato dia-mes-ano, coerente com o arquivo de origem. O parse explicito evita inversao silenciosa de dia e mes, o que corromperia a divisao temporal usada mais adiante.

In [ ]:
URL = "https://raw.githubusercontent.com/Henrique1078/mvp-ml-rain-australia/main/data/weatherAUS.csv"

df = pd.read_csv(URL, parse_dates=["Date"], dayfirst=True)
print("Formato do dataset:", df.shape)
df.head()